<a href="https://colab.research.google.com/github/chetanbheem12-pande/gcolab/blob/main/ComfyUI%20Colab%20Upgraded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ComfyUI: Professional Edition

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### 🖥️ **High-Performance AI Environment**
This notebook provides a stable, persistent, and optimized environment for running ComfyUI on Google Colab.

**Key Features:**
- **Hybrid Storage:** Executes code on the local VM for speed, but persists data (Models, Output, Nodes) to Google Drive.
- **Modern Downloader:** Python-native, high-speed downloader with visual progress bars.
- **Memory Safety:** Prevents OOM (Out of Memory) errors via selectable GPU profiles.

In [7]:
# ═══════════════════════════════════════════════════════
# CELL 1 — System Initialization
# ═══════════════════════════════════════════════════════

#@title ⚡ 1. System Initialization
#@markdown ### Setup Options
#@markdown Run this cell first every session. Takes ~1 min after first run.

UPDATE_COMFYUI = True #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}

import os, shutil, subprocess
from google.colab import drive

LOCAL = "/content/ComfyUI"
DRIVE = "/content/drive/MyDrive/ComfyUI"

# ── Mount Drive ──
print("💾 Mounting Google Drive...")
drive.mount('/content/drive')

# ── Clone or update ComfyUI ──
if not os.path.exists(f"{LOCAL}/main.py"):
    print("📦 Cloning ComfyUI (first time only)...")
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/comfyanonymous/ComfyUI", LOCAL], check=True)
else:
    if UPDATE_COMFYUI:
        print("🔄 Checking for ComfyUI updates...")
        subprocess.run(["git", "-C", LOCAL, "pull"])
    else:
        print("✅ ComfyUI found — skipping update")

# ── Symlink heavy folders → Drive (persistent storage) ──
print("🔗 Configuring persistent storage...")
for folder in ["models", "custom_nodes", "output", "input", "user"]:
    local_path = f"{LOCAL}/{folder}"
    drive_path = f"{DRIVE}/{folder}"
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(local_path):
        print(f"  ✅ {folder}/ already linked")
    elif os.path.exists(local_path):
        shutil.rmtree(local_path)
        os.symlink(drive_path, local_path)
        print(f"  ✅ {folder}/ → Drive")
    else:
        os.symlink(drive_path, local_path)
        print(f"  ✅ {folder}/ → Drive")

# ── Install ComfyUI-Manager ──
if INSTALL_COMFYUI_MANAGER:
    manager = f"{LOCAL}/custom_nodes/ComfyUI-Manager"
    if not os.path.exists(f"{manager}/__init__.py"):
        print("📦 Installing ComfyUI Manager...")
        subprocess.run(["git", "clone", "--depth=1",
            "https://github.com/ltdrdata/ComfyUI-Manager", manager], check=True)
    else:
        print("✅ ComfyUI Manager is installed.")
        subprocess.run(["git", "-C", manager, "pull"])

# ── Install Python dependencies ──
print("🛠️  Installing Python environment requirements...")
subprocess.run(["pip", "install", "gguf", "pyngrok", "-q"])
subprocess.run(["pip", "install", "xformers!=0.0.18", "-q",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121"])
subprocess.run(["pip", "install", "insightface", "onnxruntime-gpu", "-q"])

# ── Write model paths config ──
yaml_content = """comfyui:
    base_path: /content/ComfyUI/
    checkpoints: models/checkpoints/
    diffusion_models: models/diffusion_models/
    unet: models/diffusion_models/
    vae: models/vae/
    loras: models/loras/
    text_encoders: models/text_encoders/
    clip: models/text_encoders/
    upscale_models: models/upscale_models/
"""
with open(f"{LOCAL}/extra_model_paths.yaml", "w") as f:
    f.write(yaml_content)

print("\n✅ [SYSTEM READY] Initialization complete.")

💾 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Checking for ComfyUI updates...
🔗 Configuring persistent storage...
  ✅ models/ already linked
  ✅ custom_nodes/ already linked
  ✅ output/ already linked
  ✅ input/ already linked
  ✅ user/ already linked
✅ ComfyUI Manager is installed.
🛠️  Installing Python environment requirements...

✅ [SYSTEM READY] Initialization complete.


In [ ]:
#@title 📦 2. Model & Node Downloader
#@markdown ### Paste download links below (comma or newline separated)
#@markdown Existing files on Drive are **automatically skipped** — safe to re-run daily.
#@markdown
#@markdown **Pre-filled with LTX 2.3 GGUF settings for T4 GPU (15GB VRAM)**

CHECKPOINT_URLS = "" #@param {type:"string"}
UNET_DIFFUSION_URLS = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf" #@param {type:"string"}
TEXT_ENCODER_URLS = "https://huggingface.co/unsloth/gemma-3-12b-it-qat-GGUF/resolve/main/gemma-3-12b-it-qat-UD-Q4_K_XL.gguf" #@param {type:"string"}
CLIP_VISION_URLS = "" #@param {type:"string"}
VAE_URLS = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors" #@param {type:"string"}
LORA_URLS = "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "https://github.com/Lightricks/ComfyUI-LTXVideo,https://github.com/city96/ComfyUI-GGUF,https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite,https://github.com/kijai/ComfyUI-KJNodes" #@param {type:"string"}

import os, subprocess, requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

WORKSPACE = "/content/ComfyUI"

# Min file sizes to detect corrupt/incomplete downloads
MIN_SIZES = {
    "ltx-2.3-22b-dev-Q4_K_M.gguf":                   12_000_000_000,  # 12GB
    "gemma-3-12b-it-qat-UD-Q4_K_XL.gguf":             2_000_000_000,  # 2GB
    "ltx-2.3-22b-dev_video_vae.safetensors":             100_000_000,  # 100MB
    "ltx-2.3-22b-distilled-lora-384.safetensors":         10_000_000,  # 10MB
    "ltx-2.3-spatial-upscaler-x2-1.0.safetensors":       100_000_000,  # 100MB
}

DIRS = {
    "checkpoints":    f"{WORKSPACE}/models/checkpoints",
    "unet":           f"{WORKSPACE}/models/diffusion_models",
    "clip":           f"{WORKSPACE}/models/text_encoders",
    "clip_vision":    f"{WORKSPACE}/models/clip_vision",
    "vae":            f"{WORKSPACE}/models/vae",
    "loras":          f"{WORKSPACE}/models/loras",
    "controlnet":     f"{WORKSPACE}/models/controlnet",
    "upscale_models": f"{WORKSPACE}/models/upscale_models",
    "embeddings":     f"{WORKSPACE}/models/embeddings",
    "custom_nodes":   f"{WORKSPACE}/custom_nodes",
}

def get_filename(url, response):
    if "Content-Disposition" in response.headers:
        import re
        fname = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if fname: return fname[0]
    return unquote(os.path.basename(urlparse(url).path))

def download_file(url, target_dir):
    try:
        response = requests.get(url, stream=True, allow_redirects=True)
        response.raise_for_status()
        filename = get_filename(url, response)
        file_path = os.path.join(target_dir, filename)
        total_size = int(response.headers.get('content-length', 0))

        # Check if file exists and is large enough (not corrupt)
        min_size = MIN_SIZES.get(filename, 1_000_000)
        if os.path.exists(file_path):
            current_size = os.path.getsize(file_path)
            if current_size >= min_size:
                print(f"   ⏩ Already on Drive (skipping): {filename} ({current_size/1e9:.1f}GB)")
                return
            else:
                print(f"   🗑️  Corrupt file detected ({current_size/1e9:.2f}GB) — re-downloading: {filename}")
                os.remove(file_path)

        print(f"   📥 Downloading: {filename}")
        with tqdm(total=total_size, unit='B', unit_scale=True,
                  unit_divisor=1024, desc="      🚀 Progress",
                  dynamic_ncols=True) as bar:
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        final_size = os.path.getsize(file_path)
        if final_size >= min_size:
            print(f"      ✅ Download Complete ({final_size/1e9:.1f}GB)\n")
        else:
            print(f"      ⚠️  Download may be incomplete ({final_size/1e9:.2f}GB)\n")

    except Exception as e:
        print(f"   ❌ Failed to download: {url}")
        print(f"      Error: {e}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return
    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    os.makedirs(target_dir, exist_ok=True)
    print(f"\n📂 Category: {os.path.basename(target_dir)}")
    for url in url_list:
        if is_node:
            node_name = url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if os.path.exists(f"{node_path}/__init__.py"):
                print(f"   ⏩ Already installed: {node_name}")
            else:
                print(f"   ⬇️  Cloning: {node_name}...")
                subprocess.run(["git", "clone", "--depth=1", url, node_path])
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      📦 Installing requirements...")
                    subprocess.run(["pip", "install", "-r", req, "-q"])
                print(f"      ✅ Installed\n")
        else:
            download_file(url, target_dir)

process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

# ── Final status summary ──
print("\n" + "="*50)
print("📊 FINAL MODEL STATUS")
print("="*50)
checks = [
    ("diffusion_models", "ltx-2.3-22b-dev-Q4_K_M.gguf",               12.0, "Main video model"),
    ("text_encoders",    "gemma-3-12b-it-qat-UD-Q4_K_XL.gguf",         2.0, "Text encoder"),
    ("vae",              "ltx-2.3-22b-dev_video_vae.safetensors",        0.1, "VAE"),
    ("loras",            "ltx-2.3-22b-distilled-lora-384.safetensors",  0.01, "LoRA"),
    ("upscale_models",   "ltx-2.3-spatial-upscaler-x2-1.0.safetensors", 0.1, "Upscaler"),
]
all_good = True
for folder, fname, min_gb, label in checks:
    path = f"{WORKSPACE}/models/{folder}/{fname}"
    if os.path.exists(path):
        size = os.path.getsize(path)/1e9
        ok = size >= min_gb
        icon = "✅" if ok else "⚠️ TOO SMALL"
        print(f"{icon} {label}: {fname} ({size:.2f}GB)")
        if not ok: all_good = False
    else:
        print(f"❌ MISSING — {label}: {fname}")
        all_good = False

print()
print("🎉 All models ready — proceed to Cell 3!" if all_good else
      "⚠️  Fix issues above before continuing")


📂 Category: diffusion_models
   🗑️  Corrupt file detected (3.28GB) — re-downloading: ltx-2.3-22b-dev-Q4_K_M.gguf
   📥 Downloading: ltx-2.3-22b-dev-Q4_K_M.gguf


      🚀 Progress:   0%|          | 0.00/13.3G [00:00<?, ?B/s]

      ✅ Download Complete (14.3GB)


📂 Category: text_encoders
   ⏩ Already on Drive (skipping): gemma-3-12b-it-qat-UD-Q4_K_XL.gguf (2.5GB)

📂 Category: vae
   ⏩ Already on Drive (skipping): ltx-2.3-22b-dev_video_vae.safetensors (1.5GB)

📂 Category: loras
   🗑️  Corrupt file detected (0.00GB) — re-downloading: ltx-2.3-22b-distilled-lora-384.safetensors
   📥 Downloading: ltx-2.3-22b-distilled-lora-384.safetensors


      🚀 Progress:   0%|          | 0.00/7.08G [00:00<?, ?B/s]

In [2]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Session Anti-Disconnect
# ═══════════════════════════════════════════════════════

#@title 🔊 3. Session Anti-Disconnect
#@markdown **Keep-Alive Audio**
#@markdown Running this silent audio loop prevents the browser tab from sleeping.

from IPython.display import Audio, display
import numpy as np

sr = 22050
silence = np.zeros(int(sr * 600))  # 10 min loop
display(Audio(silence, rate=sr, autoplay=True))
print("🔊 Keep-alive audio running — leave this playing and proceed to Cell 4")


In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Start ComfyUI Session
# ═══════════════════════════════════════════════════════

#@title 🚀 4. Start ComfyUI Session
#@markdown ### Performance Profile
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]
#@markdown ### Visual Settings
LIVE_GENERATION_PREVIEWS = True #@param {type:"boolean"}
#@markdown ### Tunnel Settings
#@markdown Get a free token at [ngrok.com](https://ngrok.com) for stable persistent links
NGROK_TOKEN = "" #@param {type:"string"}
#@markdown ---
#@markdown **Model Info (pre-configured for LTX 2.3 GGUF on T4)**
#@markdown - Model: `ltx-2.3-22b-dev-Q4_K_M.gguf` (14GB, fits T4)
#@markdown - Resolution: 544×960 portrait (9:16 for Reels/TikTok)
#@markdown - Frames: 49 (~2 sec) — safe for 15GB VRAM

import subprocess, threading, time, socket, os
from pyngrok import ngrok

WORKSPACE = "/content/ComfyUI"
PORT = 8188

# ── Build args ──
ARGS = ""
if "Low VRAM" in MEMORY_PROFILE:
    ARGS += " --lowvram"
elif "High VRAM" in MEMORY_PROFILE:
    ARGS += " --highvram"
if LIVE_GENERATION_PREVIEWS:
    ARGS += " --preview-method auto"

# ── Install gguf (required every session) ──
print("🔧 Installing gguf module...")
subprocess.run(["pip", "install", "gguf", "-q"])
print("✅ gguf ready")

# ── ngrok tunnel ──
def start_tunnel():
    print("⏳ Waiting for ComfyUI to come online...")
    for _ in range(120):
        time.sleep(2)
        s = socket.socket()
        ok = s.connect_ex(('127.0.0.1', PORT)) == 0
        s.close()
        if ok: break

    print("\n🟢 ComfyUI is Online. Generating Access Link...\n")

    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)

    try:
        tunnel = ngrok.connect(PORT, "http")
        url = tunnel.public_url
        print("\n" + "="*60)
        print(f"🔗 ACCESS LINK: {url}")
        print("="*60)
        print("\n✅ Click the link above to open ComfyUI")
        print("📁 Videos auto-save to: Google Drive/ComfyUI/output/")
        print("⚠️  If link shows error, wait 30 seconds and refresh")
    except Exception as e:
        print(f"❌ Tunnel error: {e}")
        print("Re-run this cell to get a fresh link")

threading.Thread(target=start_tunnel, daemon=True).start()

# ── Launch ComfyUI ──
if os.path.exists(f"{WORKSPACE}/main.py"):
    os.chdir(WORKSPACE)
    print(f"🚀 Launching ComfyUI with profile: [{MEMORY_PROFILE}]...")
    os.system(f"python main.py --dont-print-server --listen 127.0.0.1 --port {PORT} {ARGS}")
else:
    print("❌ [ERROR] System files missing. Please re-run Cell 1.")

🔧 Installing gguf module...
✅ gguf ready
/content/ComfyUI
🚀 Launching ComfyUI [Standard (Auto-Detect)]...

🟢 ComfyUI is Online. Generating Access Link...


🔗 ACCESS LINK: https://5b4d-34-158-60-24.ngrok-free.app

✅ Click the link above to open ComfyUI
⚠️  If link shows error, wait 30 seconds and refresh
